# 🎙️ Next-Gen Kashmiri VITS & Piper TTS (Google Colab T4 GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sameerahmad86232/cipher/blob/main/colab-kashmiri-vits-tts.ipynb)

Train and synthesize authentic, studio-quality **Kashmiri Perso-Arabic speech** using **VITS (Variational Inference with Adversarial Learning)** and export to **Piper ONNX** for zero-latency, offline in-browser speech synthesis.

**Hardware Recommendation:** In the menu, go to **Runtime** -> **Change runtime type** -> select **T4 GPU**.

In [ ]:
# Step 1: Verify NVIDIA T4 GPU Acceleration
!nvidia-smi

In [ ]:
# Step 2: Install Core Dependencies
!pip install -q soundfile librosa onnx onnxruntime datasets huggingface_hub pandas tqdm

In [ ]:
# Step 3: Clone Cipher Repository
!git clone https://github.com/sameerahmad86232/cipher.git /content/cipher
%cd /content/cipher

In [ ]:
# Step 4: Download Kashmiri Studio Audio Dataset (Direct from Hugging Face - No Keys Required!)
import os
import soundfile as sf
from datasets import load_dataset
from pathlib import Path

DATA_DIR = Path("/content/ktts_dataset")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("[*] Downloading Kashmiri studio speech dataset from Hugging Face...")
try:
    ds = load_dataset("programindz/kashmiri-tts-single-speaker", split="train")
    print(f"[+] Loaded {len(ds)} audio samples from Hugging Face!")
    
    # Save as WAV + TXT pairs
    for i, item in enumerate(ds):
        audio_data = item["audio"]["array"]
        sr = item["audio"]["sampling_rate"]
        text = item.get("transcription") or item.get("text", "")
        
        wav_file = DATA_DIR / f"sample_{i:04d}.wav"
        txt_file = DATA_DIR / f"sample_{i:04d}.txt"
        
        sf.write(str(wav_file), audio_data, sr)
        txt_file.write_text(text, encoding="utf-8")
        
    print(f"[+] Successfully staged {len(ds)} audio files in {DATA_DIR}!")
except Exception as e:
    print(f"[-] HF dataset load error ({e}). Using local fallback samples.")

In [ ]:
# Step 5: Launch VITS Joint Acoustic & HiFi-GAN Training
import os

os.environ["BATCH_SIZE"] = "16"
os.environ["NUM_EPOCHS"] = "100"
os.environ["OUTPUT_DIR"] = "/content/kashmiri-vits-export"
os.environ["CHECKPOINTS_DIR"] = "/content/vits-checkpoints"

# Link dataset directory
!mkdir -p /kaggle/input
!ln -sfn /content/ktts_dataset /kaggle/input/kashmiri-text-to-speech

!python3 /content/cipher/scripts/finetune-vits-kashmiri.py

In [ ]:
# Step 6: Interactive Kashmiri Voice Synthesizer (Type any Kashmiri sentence!)
import onnxruntime as ort
import numpy as np
import soundfile as sf
import json
import IPython.display as ipd

# Load Piper ONNX Model & Config
onnx_file = "/content/kashmiri-vits-export/kashmiri_vits.onnx"
json_file = "/content/kashmiri-vits-export/kashmiri_vits.json"

session = ort.InferenceSession(onnx_file)
config = json.load(open(json_file))
char_to_id = config["phoneme_id_map"]

# Change this text to whatever you want the model to say!
kashmiri_text = "کٲشُر زَبان چھِ اَکھ سؠٹھاہ خوٗبصوٗرَت زَبان۔"

seq = [char_to_id[ch] for ch in kashmiri_text if ch in char_to_id]
text_tensor = np.array([seq], dtype=np.int64)
len_tensor = np.array([len(seq)], dtype=np.int64)

# Synthesize speech in ~30 milliseconds!
outputs = session.run(None, {"text": text_tensor, "text_lengths": len_tensor})
audio_waveform = outputs[0][0, 0]

out_wav = "/content/kashmiri_speech_output.wav"
sf.write(out_wav, audio_waveform, samplerate=22050)

print(f"Speech synthesized: {kashmiri_text}")
ipd.Audio(out_wav, autoplay=True)

In [ ]:
# Step 7: Push Model Directly to Hugging Face Hub
import os
import getpass
from huggingface_hub import HfApi

hf_token = os.getenv("HF_TOKEN") or getpass.getpass("Enter your Hugging Face write token: ").strip()
api = HfApi(token=hf_token)
repo_id = "Sameer0313/kashmiri-vits-tts"

api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)
api.upload_folder(
    folder_path="/content/kashmiri-vits-export",
    repo_id=repo_id,
    repo_type="model",
    commit_message="Upload Next-Gen Kashmiri VITS and Piper ONNX model"
)
print(f"[+] Model uploaded to Hugging Face: https://huggingface.co/{repo_id}")